In [6]:
!pip install edge-tts
!pip install SpeechRecognition pydub deep-translator
!apt-get install ffmpeg -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 5.3 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


# Introducción

En este laboratorio vamos a poner en práctica lo aprendido en este tema. Cosas a tener en cuenta.


*   Se puede utilizar cualquier componente siempre y cuando sea gratuito.
*   Al ser los componentes gratuitos no hay que preocuparse por la calidad de las voces.
*   Es un laboratorio guiado el paso 2 depende del paso 1, así sucesivamente.









In [14]:
nombre = "Daniel Porras Morales"
ciudad = "Tresjuncos"
texto = f"Hola soy {nombre} vivo en {ciudad} y estoy haciendo el laboratorio 3 de la asignatura de modelos."


# Definimos las voces masculinas (Neural voices)
# Alvaro es voz masculina de España
VOZ_ES_MASCULINA = "es-ES-AlvaroNeural"
# Cecilio es voz masculina de México
VOZ_MX_MASCULINA = "es-MX-GerardoNeural"

print(f"Texto a procesar: {texto}")

Texto a procesar: Hola soy Daniel Porras Morales vivo en Tresjuncos y estoy haciendo el laboratorio 3 de la asignatura de modelos.


# Paso 1:  Sintetización de audios a un mp3

Mediante un TTS tenemos que generar los siguientes ficheros de audio:


*   Hola {vuestroNombre} vivo en {ciudad donde vives} estoy haciendo el laboratorio 3 de la asignatura de modelos.
Ejemplo: Hola Daniel Porras Morales vivo en Tresjuncos y estoy haciendo el laboratorio 3 de la asignatura de modelos.

*   Sintetizar la frase anterior en ES-MX.



In [15]:
import edge_tts

# 1. Audio Español (España)
tts_es = edge_tts.Communicate(texto, VOZ_ES_MASCULINA)
file_es = "audio_hombre_espanol.mp3"
await tts_es.save(file_es)
print(f"✅ Audio generado (Español): {file_es}")

✅ Audio generado (Español): audio_hombre_espanol.mp3


In [16]:
# 2. Audio Español (México)
tts_mx = edge_tts.Communicate(texto, VOZ_MX_MASCULINA)
file_mx = "audio_hombre_mexicano.mp3"
await tts_mx.save(file_mx)
print(f"✅ Audio generado (Mexicano): {file_mx}")

NoAudioReceived: No audio was received. Please verify that your parameters are correct.

# Paso 2: Transcribir los ficheros de audio a texto.

Mediante un ASR se desea transcribir ambos ficheros a texto, es posible que además de un ASR se necesite una libreria de tratamiento de audios.

In [10]:
import speech_recognition as sr
from pydub import AudioSegment

def transcribir_audio(archivo_mp3):
    # 1. Convertir MP3 a WAV (necesario para SpeechRecognition)
    archivo_wav = archivo_mp3.replace(".mp3", ".wav")
    audio = AudioSegment.from_mp3(archivo_mp3)
    audio.export(archivo_wav, format="wav")

    # 2. Inicializar el reconocedor
    r = sr.Recognizer()

    # 3. Procesar el archivo
    with sr.AudioFile(archivo_wav) as source:
        audio_data = r.record(source) # Leer todo el archivo
        try:
            # Usamos el servicio gratuito de Google
            texto = r.recognize_google(audio_data, language="es-ES")
            return texto
        except sr.UnknownValueError:
            return "No se pudo entender el audio"
        except sr.RequestError as e:
            return f"Error en el servicio de Google; {e}"

# Ejecutamos la transcripción para ambos archivos
texto_transcrito_es = transcribir_audio("audio_hombre_español.mp3")
texto_transcrito_mx = transcribir_audio("audio_hombre_mexicano.mp3")

print("--- RESULTADOS DE LA TRANSCRIPCIÓN ---")
print(f"🇪🇸 Audio España: {texto_transcrito_es}")
print(f"🇲🇽 Audio México: {texto_transcrito_mx}")

FileNotFoundError: [Errno 2] No such file or directory: 'audio_hombre_español.mp3'

# Paso 3: Traducir al inglés.

Para este punto vamos a realizar una grabación con el texto del punto 1 en formato mp3, vamos a subir el archivo a Google Colab y lo vamos a traducir al inglés.

Subimos el fichero y realizamos la traducción al inglés.

In [ ]:
from deep_translator import GoogleTranslator
from google.colab import files



# --- MODO SIMULACIÓN ---
nombre_archivo_subido = "audio_español.mp3"
# -----------------------

print(f"Procesando archivo: {nombre_archivo_subido}")

# 2. Primero transcribimos el audio a texto (Reusamos la función del Paso 2)
texto_original = transcribir_audio(nombre_archivo_subido)
print(f"Texto detectado: {texto_original}")

# 3. Traducimos el texto al inglés
traductor = GoogleTranslator(source='auto', target='en')
texto_ingles = traductor.translate(texto_original)

print("\n--- TRADUCCIÓN FINAL ---")
print(f"🇬🇧 English: {texto_ingles}")

Procesando archivo: audio_español.mp3
Texto detectado: hola soy Daniel Porras Morales vivo en tres juncos y estoy haciendo el laboratorio 3 de la asignatura de modelos

--- TRADUCCIÓN FINAL ---
🇬🇧 English: Hello, I'm Daniel Porras Morales, I live in Tres Juncos and I'm doing laboratory 3 of the models subject.
